In [ ]:
# =========================================================================
# SVD-XT 1.1 (Stable Video Diffusion) 高清無崩壞圖生影 Worker
# =========================================================================
import os, sys, io, time, gc, json, urllib.request
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, psutil
from PIL import Image

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

print("📦 正在安裝與確認 SVD-XT 視頻生成專用依賴庫...")
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers>=0.41.0", "transformers>=5.17.0", "accelerate", "imageio[ffmpeg]"], check=False)

from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

model_id = "stabilityai/stable-video-diffusion-img2vid-xt-1-1"
print(f"🎬 [頂級肖像保真圖生影模型] 正在載入: {model_id}...")

# 啟用 FP16 精度與逐組件 CPU Offload，顯存僅需 ~6GB，零 OOM 且五官保真度極高
pipe = StableVideoDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    variant="fp16"
)
pipe.enable_model_cpu_offload()

# 載入通過 7 項品質檢驗的首幀 2K 母圖
mother_url = "https://raw.githubusercontent.com/Chenponpon/colab-ai-media-lab/main/assets/mother_images/portrait_2.jpg"
print(f"📥 正在從 GitHub 載入母圖: {mother_url} ...")
urllib.request.urlretrieve(mother_url, "input_mother.jpg")
input_img = Image.open("input_mother.jpg").convert("RGB")
orig_w, orig_h = input_img.size
print(f"   首幀原始解析度: {orig_w}x{orig_h}")

# 鎖定 SVD-XT 原生直式超清規格 (576x1024，長寬比 0.5625，高度契合直式人物視角)
vid_w, vid_h = 576, 1024
input_img = input_img.resize((vid_w, vid_h), Image.LANCZOS)
print(f"   縮放至視頻標準解析度: {vid_w}x{vid_h}")

generator = torch.manual_seed(42)

print("⏳ 正在啟動 SVD-XT 高保真圖生影管線 (25 幀 / 零溶臉零畸變電影級微動態)...")
t0 = time.time()
video_frames = pipe(
    input_img,
    height=vid_h,
    width=vid_w,
    num_frames=25,
    decode_chunk_size=4,
    motion_bucket_id=45,
    noise_aug_strength=0.02,
    generator=generator
).frames[0]

dt = round(time.time() - t0, 2)
print(f"✨ 視頻生成完畢！耗時: {dt} 秒，正在輸出 MP4...")

out_vid_name = "changing_clothes_video.mp4"
export_to_video(video_frames, out_vid_name, fps=12)
vid_sz_mb = round(os.path.getsize(out_vid_name) / (1024 * 1024), 2)
print(f"🎉 影片已成功輸出為 {out_vid_name} (體積: {vid_sz_mb} MB)！")

bench = {
    "engine": "Stable Video Diffusion (SVD-XT 1.1)",
    "gpu": gpu_name,
    "elapsed_sec": dt,
    "frames": 25,
    "fps": 12,
    "resolution": f"{vid_w}x{vid_h}",
    "video_file": out_vid_name,
    "file_size_mb": vid_sz_mb
}
with open("video_benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench, bf, indent=2)
